In [ ]:
# %pip install pymupdf pypdf langchain_community

In [ ]:
from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader('../data/OneNYC_2050_Strategic_Plan.pdf')
data_nyc = loader.load()
print(data_nyc)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
all_splits = text_splitter.split_documents(data_nyc)

In [ ]:
for i, split in enumerate(all_splits):
    print(f"Split {i+1}:--------------------------------------\n")
    print(split)

In [ ]:
print(type(all_splits[0]))

In [ ]:
loader_seoul = PyPDFLoader("../data/2040_seoul_plan.pdf")
data_seoul = loader_seoul.load()
seoul_splits = text_splitter.split_documents(data_seoul)
for i, split in enumerate(seoul_splits):
    print(f"Split {i+1}:---------------------------------------")
    print(split)

In [ ]:
print(seoul_splits[50].page_content)
print('---------------------------')
print(seoul_splits[51].page_content)

In [ ]:
for i in range(len(seoul_splits) - 1):
    seoul_splits[i].page_content += "\n" + seoul_splits[i+1].page_content[:100]

print(seoul_splits[50].page_content)
print('---------------------------')
print(seoul_splits[51].page_content)

In [ ]:
print(len(all_splits))
all_splits.extend(seoul_splits)
print(len(all_splits))

In [ ]:
%pip install langchain_chroma

In [ ]:
%pip install langchain-openai

In [ ]:
%pip install dotenv

In [6]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')
# v = embedding.embed_query("뉴욕의 온실가스 저감 정책은 뭐야?")
# print(v)
# print(len(v))

In [7]:
from langchain_chroma import Chroma
import os


persist_directory = '../chroma_store'

if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory,

    )
else:
    print("Loding existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding,
    )

Loding existing Chroma store


In [8]:
retriever = vectorstore.as_retriever(k=3)
docs = retriever.invoke("서울시의 환경 정책이 궁금해.")

for d in docs:
    print(d)
    print('-----')

page_content='제4절 기후·환경 부문1. 개요Ÿ기후변화는 21세기에 전 지구적으로 가장 위중한 영향을 미칠 것으로 예상되며, 시민 생활의 모든 측면과 연관되어 있어 향후 서울시의 적극적인 대응이 필요하다.Ÿ탄소중립 목표뿐만 아니라 미세먼지로부터 시민 건강을 지키기 위해서는 건물, 교통, 에너지 등 도시의 주요 인프라 전반의 혁신이 요구되며, 이를 위해 새로운 기술과 혁신적 제도가 필요하다. 제로에너지 건물, 친환경 차량 및 교통 인프라의 확대, 자원·에너지 순환 기반 조성으로 온실가스와 미세먼지 배출량을 획기적으로 감축해야 한다. Ÿ기후변화에 따른 폭염, 풍수해, 도심열섬현상 등 기후재난 및 극한 기후현상이 심해질 것으로 전망되어 보다 능동적인 대비가 필요하다. Ÿ한편, 환경보존과 쾌적한 도시환경을 위해 도심 곳곳 시민 모두가 누릴 수 있는 도심숲과 생활공원 등 녹색공간을 조성하고, 이를 수변 공간과 연계하여 풍부하고 지속가능한 자연환경이 확보될 수 있도록 한다.Ÿ장기적인 측면에서 시민 개개인과 기업 등 다양한 도시 내 행위자의 적극적인 협조가 필수적이며 이를 위해 중앙정부와 서울시 환경계획 담당부서와의 협력적이고 포용적인 거버넌스 체계를 구축하도록 한다.목표 전략3-12050 탄소중립 실현을 위한 도시 인프라 전환3-1-1건물 부문의 탄소배출을 감축하기 위한 친환경 기술 개발 및 적극 적용3-1-2미래 모빌리티 기술 활용과 친환경 수송 차량 및 관련 인프라 확충3-1-3에너지 전환을 위한 청정에너지 기반 구축3-1-4대기 환경을 고려한 공간계획과 배출원 관리체계 강화3-2건강한 순환도시 조성을 위한자립적인 자원순환 체계 구축3-2-1자원순환·관리 자립을 위한 분산형 폐기물처리 시설 구축3-2-2기후 행동 포용적 거버넌스 구축을 위한 시민 행동 활성화3-3사람과 자연의 공존을 위한친환경 생태도시 구축3-3-1건물 에너지 분야 효율성 개선 및 도심 속 생물 다양성 확보3-3-2지속가능한 통합 물순환 체계 구축3-4다양한 수변을 경험할 수 있는수변감성도

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI



chat = ChatOpenAI(model='gpt-4o-mini')

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name='messages'),
    ]
)

document_chain = create_stuff_documents_chain(chat, question_answering_prompt)

In [10]:
from langchain_classic.memory import ChatMessageHistory


# 채팅 메시지를 저장할 메모리 객체 생성
chat_history = ChatMessageHistory()
# 사용자 질문을 메모리에 저장
chat_history.add_user_message("서울시의 온실가스 저감 정책에 대해 알려줘.")

# 문서 검색하고 답변 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

chat_history.add_ai_message(answer)

print(answer)

서울시는 온실가스 저감을 위한 여러 정책을 추진하고 있으며, 주요 내용은 다음과 같습니다:

1. **탄소중립 목표**: 서울시는 2050 탄소중립 실현을 목표로 하고 있으며, 기후변화에 적극 대응하기 위한 다양한 노력을 기울이고 있습니다.

2. **친환경 건물 및 교통 인프라 확장**: 서울시는 제로에너지 건물과 친환경 차량을 도입하고 관련 교통 인프라를 확충하여 온실가스 배출을 감소시키고 있습니다. 건물 부문의 탄소배출 감축을 위한 친환경 기술 개발 및 적용이 이루어지고 있습니다.

3. **청정에너지 기반 구축**: 에너지 전환을 위해 청정에너지 기반을 마련하고 있으며, 소규모 분산형 발전시설 확대를 지원하는 등의 노력을 하고 있습니다.

4. **자원순환 체계 구축**: 자원순환을 위해 분산형 폐기물 처리 시설을 구축하고, 생활폐기물의 재활용을 최대화하기 위한 정책과 기술 개발을 추진하고 있습니다.

5. **시민 참여 및 교육**: 시민들이 기후 행동에 참여할 수 있도록 행동을 유도하고, 기후 변화에 대한 이해를 높이기 위해 다양한 친환경 교육 프로그램을 운영하고 있습니다.

6. **대기 환경 관리**: 대기질 개선을 위해 배출원 관리 체계를 강화하고, PM2.5 및 NOx와 같은 대기오염물질의 배출을 감축하기 위한 서울시 맞춤형 대책도 추진됩니다.

7. **친환경 생태도시 구축**: 자연 생태계를 보존하고 도심 속 생물 다양성을 확보하기 위한 노력을 강화하며, 지속 가능한 물 순환 체계를 구축하여 기후 변화에 더욱 적극적으로 대응하고 있습니다.

이러한 정책들은 서울시가 직면한 기후 재난과 극한 기후 현상에 대한 보다 능동적인 대응을 위해 필요한 전략들입니다.


In [11]:
for m in chat_history.messages:
    print(m)

content='서울시의 온실가스 저감 정책에 대해 알려줘.' additional_kwargs={} response_metadata={}
content='서울시는 온실가스 저감을 위한 여러 정책을 추진하고 있으며, 주요 내용은 다음과 같습니다:\n\n1. **탄소중립 목표**: 서울시는 2050 탄소중립 실현을 목표로 하고 있으며, 기후변화에 적극 대응하기 위한 다양한 노력을 기울이고 있습니다.\n\n2. **친환경 건물 및 교통 인프라 확장**: 서울시는 제로에너지 건물과 친환경 차량을 도입하고 관련 교통 인프라를 확충하여 온실가스 배출을 감소시키고 있습니다. 건물 부문의 탄소배출 감축을 위한 친환경 기술 개발 및 적용이 이루어지고 있습니다.\n\n3. **청정에너지 기반 구축**: 에너지 전환을 위해 청정에너지 기반을 마련하고 있으며, 소규모 분산형 발전시설 확대를 지원하는 등의 노력을 하고 있습니다.\n\n4. **자원순환 체계 구축**: 자원순환을 위해 분산형 폐기물 처리 시설을 구축하고, 생활폐기물의 재활용을 최대화하기 위한 정책과 기술 개발을 추진하고 있습니다.\n\n5. **시민 참여 및 교육**: 시민들이 기후 행동에 참여할 수 있도록 행동을 유도하고, 기후 변화에 대한 이해를 높이기 위해 다양한 친환경 교육 프로그램을 운영하고 있습니다.\n\n6. **대기 환경 관리**: 대기질 개선을 위해 배출원 관리 체계를 강화하고, PM2.5 및 NOx와 같은 대기오염물질의 배출을 감축하기 위한 서울시 맞춤형 대책도 추진됩니다.\n\n7. **친환경 생태도시 구축**: 자연 생태계를 보존하고 도심 속 생물 다양성을 확보하기 위한 노력을 강화하며, 지속 가능한 물 순환 체계를 구축하여 기후 변화에 더욱 적극적으로 대응하고 있습니다.\n\n이러한 정책들은 서울시가 직면한 기후 재난과 극한 기후 현상에 대한 보다 능동적인 대응을 위해 필요한 전략들입니다.' additional_kwargs={} response_metadata={} tool_calls=[] in

In [12]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
query_for_nyc = "뉴욕은?"